In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.integrate
from tqdm import tqdm, trange

plt.style.use("../style.mplstyle")

In [ ]:
def derivatives(t: float, v: np.ndarray, eta: float) -> np.ndarray:
    vp = np.zeros_like(v)

    # Envelope
    r, rp = v[0:2]
    
    vp[0] = rp
    vp[1] = -r + (eta**2 / r**3) + ((1.0 - eta**2) / r)

    # Particle
    x, xp = v[2:]
    if abs(x) < r:
        f_sc = x / r**2
    else:
        f_sc = 1.0 / x

    vp[2] = xp
    vp[3] = -x + (1.0 - eta**2) * f_sc
    return vp

In [ ]:
eta = 0.5  # Tune-depression ratio
r0 = 0.62  # Initial core radius (mismatch parameter)
x0 = 0.77  # Initial particle displacement
xp0 = 0.0  # Initial particle divergence (velocity)
t_max = 160.0  # Maximum dimensionless time
t_steps = 5000

k0 = 1.0
k = eta * k0
k_env = np.sqrt(2.0 * (k**2 + k0**2))
lambda_env = 2.0 * np.pi / k_env

t_eval = np.linspace(0.0, t_max, t_steps + 1)
solution = scipy.integrate.solve_ivp(
    derivatives, 
    t_span=[0, t_max],
    t_eval=t_eval,
    y0=[r0, 0.0, x0, xp0],
    args=(eta,),
    method="LSODA", 
    rtol=1e-8, 
    atol=1e-8,
)

t = solution.t
r = solution.y[0]
x = solution.y[2]
xp = solution.y[3]

# Plot: trajectory vs. t
fig, ax = plt.subplots(figsize=(6, 3))
ax.fill_between(t, -r, +r, color="black", alpha=1.0, ec="none")
ax.plot(t, x, color="red")
ax.set_xlabel(r"$k_0 s$")
ax.set_ylabel(r"$x / r_0$")
plt.show()

# # --- Plotting Figure 2: Continuous Phase Space ---
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(x, xp, color="black", lw=0.5)
ax.set_xlabel(r'$x / r_0$')
ax.set_ylabel(r"$x' / k_0 r_0$")
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
plt.show()

In [ ]:
eta = 0.5
r0 = 0.62
xp0 = 0.0 

# Find time of envelope minimum
# [...]

k0 = 1.0
k = eta * k0
k_env = np.sqrt(2.0 * (eta**2 + 1.0))
lambda_env = 2.0 * np.pi / k_env

nturns = 1000
nparts = 16

init_coords = []
init_coords += [(x, 0.0) for x in np.linspace(0.1, 3.0, nparts // 2)]
# init_coords += [(0.0, x) for x in np.linspace(0.1, 4.0, nparts // 2)]
# init_coords = [(1, 0.0)]

t_off = 0.

coords_tbt = np.zeros((nturns + 1, nparts, 2))
for index, (x0, xp0) in enumerate(tqdm(init_coords)):    
    v = np.array([r0, 0.0, x0, xp0])

    coords_tbt[0, index, :] = [x0, xp0]
    for turn in range(nturns):
        solution = scipy.integrate.solve_ivp(
            derivatives, 
            t_span=[t_off, lambda_env + t_off],
            y0=v,
            args=(eta,),
            method="LSODA", 
            rtol=1e-8, 
            atol=1e-8,
        )
        coords_tbt[turn + 1, index, :] = solution.y[2:, -1].copy()
        v = solution.y[:, -1].copy()

In [ ]:
fig, ax = plt.subplots()
for index in range(7):
    ax.scatter(
        coords_tbt[:, index, 0],
        coords_tbt[:, index, 1],
        color="black",
        ec="none",
        s=2,
    )
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5);